# Upscale image

In [ ]:
#@title Connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title Upscale
file_path = "" #@param {type:"string"}
scale = "4x" #@param ["4x", "2x"]

# Accept paths as Drive shows them (Temp\video.mp4) or as full /content paths.
file_path = file_path.strip().strip('"').replace("\\", "/")
if file_path and not file_path.startswith("/"):
    file_path = "/content/drive/MyDrive/" + file_path.removeprefix("MyDrive/").removeprefix("My Drive/")

import os, urllib.request
if not file_path:
    raise SystemExit("Enter a file path.")
if not os.path.isfile(file_path):
    raise SystemExit(f"Not a file: {file_path}")
!pip install -q spandrel
import numpy as np, torch
from PIL import Image
from spandrel import ModelLoader

weights_url = {
    "4x": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
    "2x": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth",
}[scale]
weights = f"/content/{os.path.basename(weights_url)}"
if not os.path.exists(weights):
    urllib.request.urlretrieve(weights_url, weights)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ModelLoader().load_from_file(weights).to(device).eval()
s = model.scale

img = Image.open(file_path).convert("RGB")
x = torch.from_numpy(np.asarray(img)).permute(2, 0, 1).float().div(255).unsqueeze(0)
_, _, h, w = x.shape
out = torch.zeros(1, 3, h * s, w * s)
# Tiles keep large photos within GPU memory; the overlap hides tile seams.
tile, pad = 512, 16
with torch.no_grad():
    for y0 in range(0, h, tile):
        for x0 in range(0, w, tile):
            y1, x1 = min(y0 + tile, h), min(x0 + tile, w)
            ya, xa = max(y0 - pad, 0), max(x0 - pad, 0)
            yb, xb = min(y1 + pad, h), min(x1 + pad, w)
            t = model(x[:, :, ya:yb, xa:xb].to(device)).cpu()
            oy, ox = (y0 - ya) * s, (x0 - xa) * s
            out[:, :, y0 * s:y1 * s, x0 * s:x1 * s] = t[:, :, oy:oy + (y1 - y0) * s, ox:ox + (x1 - x0) * s]
result = (out[0].clamp(0, 1).permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)
output = f"{os.path.splitext(file_path)[0]}-{scale}.png"
Image.fromarray(result).save(output)
print(f"✅ {output} ({w}x{h} -> {w * s}x{h * s})")
